# Capstone Part 2: Supervised Fine-Tuning (SFT)

In this notebook we take our pretrained model from Part 1 and fine-tune it on instruction-response
pairs. This transforms a raw language model into one that can follow instructions.

**What SFT does:**
- Teaches the model the instruction-following format
- Conditions generation on user instructions
- Improves response quality and relevance

**What SFT does NOT do:**
- Align preferences (that is DPO, Part 3)
- Make the model refuse harmful requests (needs RLHF/DPO)

By the end you will have an instruction-tuned model that responds to prompts.

## 1. Environment Setup

In [ ]:
!pip install -q torch datasets tiktoken matplotlib tqdm trl transformers

In [ ]:
import math
import time
import json
import copy
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Model Architecture

We re-define the model architecture (same as Part 1) and load the pretrained checkpoint.
In practice you would import from a shared module; here we repeat for self-containment.

In [ ]:
# ---- Model definition (identical to 01-pretrain.ipynb) ----

@dataclass
class ModelConfig:
    vocab_size: int = 50257
    max_seq_len: int = 512
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 688
    dropout: float = 0.1
    rope_theta: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x * norm).type_as(x) * self.weight


def precompute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, freqs)
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x, freqs_cis):
    batch, seq_len, n_heads, head_dim = x.shape
    x_complex = torch.view_as_complex(x.float().reshape(batch, seq_len, n_heads, -1, 2))
    freqs_cis = freqs_cis[:seq_len].unsqueeze(0).unsqueeze(2)
    x_rotated = torch.view_as_real(x_complex * freqs_cis).reshape(batch, seq_len, n_heads, head_dim)
    return x_rotated.type_as(x)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

    def _repeat_kv(self, x):
        if self.n_rep == 1:
            return x
        b, s, n, d = x.shape
        return x[:, :, :, None, :].expand(b, s, n, self.n_rep, d).reshape(b, s, self.n_heads, d)

    def forward(self, x, freqs_cis, mask=None):
        b, s, _ = x.shape
        q = self.wq(x).reshape(b, s, self.n_heads, self.head_dim)
        k = self.wk(x).reshape(b, s, self.n_kv_heads, self.head_dim)
        v = self.wv(x).reshape(b, s, self.n_kv_heads, self.head_dim)
        q, k = apply_rope(q, freqs_cis), apply_rope(k, freqs_cis)
        k, v = self._repeat_kv(k), self._repeat_kv(v)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = self.attn_dropout(F.softmax(attn, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).reshape(b, s, -1)
        return self.resid_dropout(self.wo(out))


class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w2 = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.w3 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model)
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config)

    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        head_dim = config.d_model // config.n_heads
        self.register_buffer("freqs_cis", precompute_rope_frequencies(head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None):
        b, s = input_ids.shape
        mask = torch.tril(torch.ones(s, s, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        x = self.dropout(self.tok_emb(input_ids))
        for layer in self.layers:
            x = layer(x, self.freqs_cis, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            next_token = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
        return input_ids


print("Model architecture defined.")

## 3. Load Pretrained Checkpoint

In [ ]:
checkpoint_path = Path("checkpoints/best_pretrained.pt")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

config = checkpoint["config"]
model = MiniLLM(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

tokenizer = tiktoken.get_encoding("gpt2")

print(f"Loaded pretrained checkpoint from step {checkpoint['step']}")
print(f"Pretrained val loss: {checkpoint['val_loss']:.4f}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Instruction Dataset

For SFT, we need instruction-response pairs. We create a curated set of simple instructions
that match our model's capability (short stories, simple Q&A) plus load a subset from Alpaca.

The key design choice is the **chat template**. We use a simple format:
```
<|instruction|> {instruction} <|response|> {response} <|end|>
```

During SFT, we only compute loss on the **response tokens** (not the instruction), so the
model learns to generate responses, not to parrot instructions.

In [ ]:
# Hand-crafted instruction examples matching our model's domain (simple stories)
MANUAL_INSTRUCTIONS = [
    {"instruction": "Write a short story about a cat.",
     "response": "Once upon a time, there was a little cat named Whiskers. Whiskers loved to play in the garden. One day, she found a butterfly and chased it all around. At the end of the day, Whiskers was tired but happy. She curled up on her bed and fell asleep."},
    {"instruction": "Tell me a story about a dog who goes to the park.",
     "response": "Buddy was a happy dog who loved the park. Every morning, his owner would take him for a walk. At the park, Buddy would run and play with other dogs. He especially liked to fetch the ball. After playing, Buddy would drink water and rest under a big tree."},
    {"instruction": "Write a story about a child learning to ride a bike.",
     "response": "Lily wanted to ride a bike like the big kids. Her dad held the back of the seat while she pedaled. At first, she was scared and wobbled a lot. But she kept trying every day. One sunny morning, her dad let go and Lily rode all by herself! She was so proud and happy."},
    {"instruction": "Tell a story about friendship.",
     "response": "Tom and Sam were best friends. They did everything together. One day, Tom lost his favorite toy. He was very sad. Sam helped him look everywhere. They finally found it under the bed. Tom gave Sam a big hug. That is what friends do, they help each other."},
    {"instruction": "Write about a rainy day.",
     "response": "It was raining outside. Emma could not go out to play. She felt sad at first. Then her mom said they could bake cookies together. Emma mixed the dough and put chocolate chips in. The cookies were delicious. Emma learned that rainy days can be fun too."},
    {"instruction": "What happens when you plant a seed?",
     "response": "When you plant a seed in the ground and give it water, it starts to grow. First, a tiny root grows down into the soil. Then a small green shoot pushes up through the dirt. With sunlight and water, the shoot grows into a plant. It might even grow flowers or fruit."},
    {"instruction": "Write a bedtime story.",
     "response": "The stars came out one by one in the dark sky. Little Bear looked up and counted them. One, two, three, four, five. His mama bear said it was time for bed. Little Bear yawned and closed his eyes. He dreamed of playing in the meadow with his friends. Good night, Little Bear."},
    {"instruction": "Tell a story about the ocean.",
     "response": "Deep in the blue ocean, there lived a small fish named Finn. Finn loved to explore. He swam past colorful coral and played with the sea turtles. One day, he found a shiny shell on the ocean floor. He brought it back to show his family. They all thought it was beautiful."},
    {"instruction": "Write about a birthday party.",
     "response": "Today was Mia's birthday! She was turning five. Her mom made a big cake with pink frosting. All her friends came to the party. They played games and ate cake. Mia opened her presents and got a teddy bear. It was the best birthday ever."},
    {"instruction": "Tell a story about being brave.",
     "response": "Jack was afraid of the dark. Every night, he would hide under his blanket. One night, he heard a strange noise. He was scared but decided to be brave. He turned on his flashlight and looked around. It was just his cat playing with a toy. Jack laughed and was not scared anymore."},
]

print(f"Manual instruction examples: {len(MANUAL_INSTRUCTIONS)}")

In [ ]:
# Load additional instruction data from Alpaca (simplified subset)
from datasets import load_dataset

alpaca = load_dataset("tatsu-lab/alpaca", split="train[:500]")

# Filter to short, simple examples that match our model's capability
alpaca_instructions = []
for example in alpaca:
    # Skip examples with long inputs or outputs
    instruction = example["instruction"]
    response = example["output"]
    if example["input"]:  # Skip examples requiring additional input context
        continue
    if len(response.split()) > 150 or len(response.split()) < 10:
        continue
    alpaca_instructions.append({"instruction": instruction, "response": response})

print(f"Filtered Alpaca examples: {len(alpaca_instructions)}")

# Combine datasets
all_instructions = MANUAL_INSTRUCTIONS + alpaca_instructions
print(f"Total SFT examples: {len(all_instructions)}")
print(f"\nSample:\n  Instruction: {all_instructions[0]['instruction']}")
print(f"  Response: {all_instructions[0]['response'][:100]}...")

## 5. SFT Dataset with Masked Loss

The critical detail in SFT is **loss masking**: we only compute cross-entropy on the
response tokens, not the instruction tokens. This prevents the model from "wasting"
capacity learning to reproduce instructions.

```
Tokens:  <|instruction|> Write a story <|response|> Once upon a time ... <|end|>
Loss:    -1             -1    -1  -1   -1           LOSS LOSS LOSS ... LOSS
```

In [ ]:
# Define special token strings (we encode them as regular text since we use tiktoken)
INSTRUCTION_PREFIX = "<|instruction|> "
RESPONSE_PREFIX = " <|response|> "
END_TOKEN = " <|end|>"


class SFTDataset(Dataset):
    """SFT dataset that masks instruction tokens from loss computation."""

    def __init__(self, examples, tokenizer, max_len=512):
        self.samples = []
        self.tokenizer = tokenizer

        for ex in examples:
            # Format: <|instruction|> {instruction} <|response|> {response} <|end|>
            full_text = INSTRUCTION_PREFIX + ex["instruction"] + RESPONSE_PREFIX + ex["response"] + END_TOKEN
            prompt_text = INSTRUCTION_PREFIX + ex["instruction"] + RESPONSE_PREFIX

            full_tokens = tokenizer.encode(full_text)
            prompt_tokens = tokenizer.encode(prompt_text)

            if len(full_tokens) > max_len:
                full_tokens = full_tokens[:max_len]

            # Create input/target with masking
            input_ids = full_tokens[:-1]
            target_ids = full_tokens[1:]

            # Mask instruction tokens (set to -1 so cross_entropy ignores them)
            labels = [-1] * (len(prompt_tokens) - 1) + target_ids[len(prompt_tokens) - 1:]
            labels = labels[:len(input_ids)]  # Ensure same length

            # Pad to max_len
            pad_len = max_len - len(input_ids) - 1
            if pad_len > 0:
                input_ids = input_ids + [0] * pad_len
                labels = labels + [-1] * pad_len

            self.samples.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]["input_ids"], self.samples[idx]["labels"]


sft_dataset = SFTDataset(all_instructions, tokenizer, max_len=config.max_seq_len)
print(f"SFT dataset size: {len(sft_dataset)}")

# Inspect a sample
sample_input, sample_label = sft_dataset[0]
n_masked = (sample_label == -1).sum().item()
n_active = (sample_label != -1).sum().item()
print(f"Sample input length: {sample_input.shape[0]}")
print(f"Masked tokens (instruction): {n_masked}")
print(f"Active tokens (response): {n_active}")
print(f"Loss mask ratio: {n_active / (n_masked + n_active):.1%}")

In [ ]:
# Train/val split
val_size = max(10, len(sft_dataset) // 10)
train_size = len(sft_dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    sft_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Train: {train_size}, Val: {val_size}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 6. Pre-SFT Baseline

Before fine-tuning, let's see how the pretrained model handles instructions. It should
produce text but not follow the instruction format.

In [ ]:
def generate_response(model, tokenizer, instruction, device, max_tokens=100):
    """Generate a response given an instruction."""
    prompt = INSTRUCTION_PREFIX + instruction + RESPONSE_PREFIX
    input_ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    output_ids = model.generate(input_ids, max_new_tokens=max_tokens, temperature=0.7, top_k=40)
    full_text = tokenizer.decode(output_ids[0].tolist())
    # Extract just the response part
    if "<|response|>" in full_text:
        response = full_text.split("<|response|>")[-1]
        if "<|end|>" in response:
            response = response.split("<|end|>")[0]
        return response.strip()
    return full_text[len(prompt):].strip()


test_instructions = [
    "Write a short story about a rabbit.",
    "Tell me about the moon.",
    "Write a story about making a new friend.",
]

print("=" * 70)
print("PRE-SFT BASELINE (Pretrained model with instruction prompts)")
print("=" * 70)

model.eval()
baseline_responses = {}
for instr in test_instructions:
    resp = generate_response(model, tokenizer, instr, device)
    baseline_responses[instr] = resp
    print(f"\nInstruction: {instr}")
    print(f"Response: {resp[:200]}")
    print("-" * 70)

## 7. SFT Training Configuration

SFT uses a **lower learning rate** than pretraining to avoid catastrophic forgetting.
We also train for fewer epochs since the dataset is small and we just need to
teach the format.

In [ ]:
@dataclass
class SFTTrainConfig:
    num_epochs: int = 5
    learning_rate: float = 5e-5      # Lower than pretraining
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    max_grad_norm: float = 1.0
    log_interval: int = 10

sft_config = SFTTrainConfig()

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=sft_config.learning_rate,
    weight_decay=sft_config.weight_decay,
    betas=(0.9, 0.95),
)

total_steps = len(train_loader) * sft_config.num_epochs
warmup_steps = int(total_steps * sft_config.warmup_ratio)


def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"SFT training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Learning rate: {sft_config.learning_rate}")

## 8. SFT Training Loop

In [ ]:
@torch.no_grad()
def evaluate_sft(model, val_loader, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item()
        total_batches += 1
    model.train()
    return total_loss / max(1, total_batches)

In [ ]:
train_losses = []
val_losses_history = []
global_step = 0
best_val_loss = float("inf")
save_dir = Path("checkpoints")
save_dir.mkdir(exist_ok=True)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler(enabled=use_amp)

print(f"Starting SFT for {sft_config.num_epochs} epochs")
print("=" * 70)

model.train()
for epoch in range(sft_config.num_epochs):
    epoch_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            _, loss = model(x, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), sft_config.max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        train_losses.append(loss.item())
        epoch_loss += loss.item()
        global_step += 1

        if global_step % sft_config.log_interval == 0:
            print(f"  Step {global_step}/{total_steps} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # End of epoch evaluation
    val_loss = evaluate_sft(model, val_loader, device)
    val_losses_history.append(val_loss)
    epoch_time = time.time() - epoch_start
    avg_epoch_loss = epoch_loss / len(train_loader)

    print(f"\nEpoch {epoch+1}/{sft_config.num_epochs} | Train Loss: {avg_epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": config,
            "step": global_step,
            "val_loss": val_loss,
            "stage": "sft",
        }, save_dir / "best_sft.pt")
        print(f"  >> Saved best SFT checkpoint (val_loss={val_loss:.4f})")
    print("=" * 70)

# Save final
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config,
    "step": global_step,
    "stage": "sft",
    "train_losses": train_losses,
}, save_dir / "final_sft.pt")
print("SFT training complete!")

## 9. SFT Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Training loss
ax = axes[0]
ax.plot(train_losses, alpha=0.3, color='blue')
# Smoothed
window = min(20, len(train_losses) // 5)
if window > 1:
    smoothed = [sum(train_losses[max(0,i-window):i+1]) / min(i+1, window) for i in range(len(train_losses))]
    ax.plot(smoothed, color='blue', linewidth=2, label='Smoothed')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('SFT Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Validation loss per epoch
ax = axes[1]
ax.plot(range(1, len(val_losses_history) + 1), val_losses_history, 'ro-', markersize=8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Loss')
ax.set_title('SFT Validation Loss')
ax.grid(True, alpha=0.3)

# Loss distribution at start vs end
ax = axes[2]
n = len(train_losses) // 5
ax.hist(train_losses[:n], bins=30, alpha=0.5, label='First 20%', color='red')
ax.hist(train_losses[-n:], bins=30, alpha=0.5, label='Last 20%', color='green')
ax.set_xlabel('Loss')
ax.set_ylabel('Count')
ax.set_title('Loss Distribution: Start vs End')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('SFT Training Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'sft_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Post-SFT Generation

Now let's test the same instructions and see how the SFT model compares.

In [ ]:
model.eval()

print("=" * 70)
print("POST-SFT GENERATION")
print("=" * 70)

sft_responses = {}
for instr in test_instructions:
    resp = generate_response(model, tokenizer, instr, device)
    sft_responses[instr] = resp
    print(f"\nInstruction: {instr}")
    print(f"Response: {resp[:300]}")
    print("-" * 70)

## 11. Side-by-Side Comparison: Base vs SFT

In [ ]:
print("=" * 80)
print("COMPARISON: Base Pretrained vs SFT")
print("=" * 80)

for instr in test_instructions:
    print(f"\nInstruction: {instr}")
    print(f"\n  [BASE]  {baseline_responses[instr][:200]}")
    print(f"\n  [SFT]   {sft_responses[instr][:200]}")
    print("-" * 80)

## 12. Response Length Analysis

SFT models typically produce more structured, appropriately-lengthed responses.

In [ ]:
# Generate more responses for statistical comparison
additional_instructions = [
    "Write about a sunny day.",
    "Tell a story about a bird.",
    "Write about a trip to the beach.",
    "Tell me about a magic garden.",
    "Write a story about sharing.",
]

# Load base model for comparison
base_model = MiniLLM(config).to(device)
base_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
base_model.load_state_dict(base_checkpoint["model_state_dict"])
base_model.eval()

base_lengths = []
sft_lengths = []

all_test = test_instructions + additional_instructions
for instr in all_test:
    base_resp = generate_response(base_model, tokenizer, instr, device)
    sft_resp = generate_response(model, tokenizer, instr, device)
    base_lengths.append(len(base_resp.split()))
    sft_lengths.append(len(sft_resp.split()))

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(all_test))
width = 0.35
ax.bar([i - width/2 for i in x], base_lengths, width, label='Base', color='salmon')
ax.bar([i + width/2 for i in x], sft_lengths, width, label='SFT', color='skyblue')
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}" for i in x])
ax.set_ylabel('Response Length (words)')
ax.set_title('Response Length: Base vs SFT')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

del base_model  # Free memory
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 13. Checkpoint Verification

In [ ]:
# Verify SFT checkpoint
sft_ckpt = torch.load(save_dir / "best_sft.pt", map_location=device, weights_only=False)
print("SFT Checkpoint:")
print(f"  Stage: {sft_ckpt['stage']}")
print(f"  Step: {sft_ckpt['step']}")
print(f"  Val Loss: {sft_ckpt['val_loss']:.4f}")
print(f"  State dict keys: {len(sft_ckpt['model_state_dict'])}")

for f in save_dir.glob("*sft*"):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

## Summary

In this notebook we:

1. **Loaded** the pretrained checkpoint from Part 1
2. **Prepared** an instruction dataset with proper loss masking
3. **Trained** SFT with a lower learning rate and cosine schedule
4. **Compared** base vs SFT responses side-by-side
5. **Saved** the SFT checkpoint for DPO alignment in Part 3

The SFT model should now follow the instruction format and produce more relevant responses.
However, it may still produce low-quality or irrelevant content sometimes. DPO alignment
in the next notebook will address this.

**Checkpoint location:** `checkpoints/best_sft.pt`